**Create Dataset**

In [2]:
TRAINING_DATA_PATH = "training_data/synth_train_data.npz"

In [4]:
import librosa
import torch
import numpy as np
from helper_functions import AudioRecordingDataset, generate_dataset

In [ ]:
X, Y, HV = generate_dataset()
np.savez(file = TRAINING_DATA_PATH, X = X,Y = Y, hv = HV)

**Load Torch Dataset/Dataloader**

In [5]:
from torch.utils.data import DataLoader, Subset
import torch.nn as nn
import torch.optim as optim

data = AudioRecordingDataset(TRAINING_DATA_PATH)
val_mask = (data.hv == 0.25)
val_idx = np.where(val_mask)[0]
train_idx = np.where(~val_mask)[0]


train_X = data.X[train_idx]
data.mean = train_X.mean(axis = 0)
data.std = train_X.std(axis = 0) + 1e-8 #for 0 values

print(f"data_mean: {data.mean.shape}| data_std: {data.std.shape}")

train_data = Subset(data, train_idx)
val_data = Subset(data, val_idx)

print(f"train {len(train_data)} | val {len(val_data)}")

train_loader = DataLoader(
    dataset = train_data,
    batch_size = 32,
    shuffle = True
)
val_loader = DataLoader(
    dataset = val_data,
    batch_size = 32,
    shuffle = True
)

# batch size x feature_size (64, 84)
model = nn.Sequential(
    nn.Linear(84, 125), 
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(125, 88) # 88 = num of valid midi_notes
)
# batch size * output_dim (64, 88)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

data_mean: (84,)| data_std: (84,)
train 1760 | val 440


In [ ]:
num_epochs = 35

for epoch in range(num_epochs):
    model.train() 
    total_loss = 0
    for batch_idx, (batch_X, batch_Y) in enumerate(train_loader):
        logits = model(batch_X)
        loss = criterion(logits, batch_Y)
        #back pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    val_loss, correct, total = 0,0,0
    with torch.no_grad():
        for batch_X, batch_Y in val_loader:
            logits = model(batch_X)
            val_loss += criterion(logits, batch_Y).item()
            total += batch_Y.size(0)
            correct += (logits.argmax(dim=1) == batch_Y).sum().item()

    print(f"Epoch {epoch+1:3d} | train {total_loss/len(train_loader):.3f} "
          f"| val {val_loss/len(val_loader):.3f}| acc {correct/total:.3f}")


In [7]:
# Save the Model
checkpoint = {
    'model_state': model.state_dict(),
    'mean' : torch.tensor(data.mean, dtype=torch.float32),
    'std' : torch.tensor(data.std, dtype=torch.float32)
}

torch.save(checkpoint, 'models/linear_synth.pth')

Train On NSynth Dataset; Same linear regression model

In [ ]:
from load_validation import load_data
from sklearn.model_selection import train_test_split


X,Y,meta=load_data(path="nsynth-valid")


In [10]:
from sklearn.model_selection import GroupShuffleSplit
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

gss = GroupShuffleSplit(n_splits=1, test_size= 0.2, random_state=42)

train_idx, test_idx = next(gss.split(X, Y, meta['instrument']))
X_train, X_test = X[train_idx], X[test_idx]
Y_train, Y_test = Y[train_idx], Y[test_idx]

train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis = 0) + 1e-8

X_train = (X_train - train_mean) / train_std
X_test = (X_test - train_mean) / train_std

train_data = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(Y_train, dtype=torch.long),
    )
test_data  = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(Y_test, dtype=torch.long),
    )


train_loader = DataLoader(
    dataset = train_data,
    batch_size = 32,
    shuffle = True
)
test_loader = DataLoader(
    dataset = test_data,
    batch_size = 32,
    shuffle = True
)

model = nn.Sequential(
    nn.Linear(84, 125), 
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(125, 88) # 88 = num of valid midi_notes
)
# batch size * output_dim (64, 88)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

In [11]:
num_epochs = 20

for epoch in range(num_epochs):
    model.train() 
    total_loss = 0
    for batch_idx, (batch_X, batch_Y) in enumerate(train_loader):
        logits = model(batch_X)
        loss = criterion(logits, batch_Y)
        #back pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    val_loss, correct, total = 0,0,0
    with torch.no_grad():
        for batch_X, batch_Y in test_loader:
            logits = model(batch_X)
            val_loss += criterion(logits, batch_Y).item()
            total += batch_Y.size(0)
            correct += (logits.argmax(dim=1) == batch_Y).sum().item()


            #add overfitting check

    print(f"Epoch {epoch+1:3d} | train {total_loss/len(train_loader):.3f} "
          f"| val {val_loss/len(test_loader):.3f}| acc {correct/total:.3f}")


Epoch   1 | train 2.956 | val 1.542| acc 0.745
Epoch   2 | train 1.278 | val 0.837| acc 0.868
Epoch   3 | train 0.828 | val 0.577| acc 0.907
Epoch   4 | train 0.635 | val 0.469| acc 0.920
Epoch   5 | train 0.532 | val 0.404| acc 0.924
Epoch   6 | train 0.461 | val 0.357| acc 0.930
Epoch   7 | train 0.403 | val 0.321| acc 0.931
Epoch   8 | train 0.375 | val 0.297| acc 0.935
Epoch   9 | train 0.342 | val 0.296| acc 0.929
Epoch  10 | train 0.317 | val 0.281| acc 0.929
Epoch  11 | train 0.287 | val 0.286| acc 0.919
Epoch  12 | train 0.273 | val 0.264| acc 0.925
Epoch  13 | train 0.259 | val 0.274| acc 0.927
Epoch  14 | train 0.247 | val 0.264| acc 0.919
Epoch  15 | train 0.244 | val 0.275| acc 0.920
Epoch  16 | train 0.228 | val 0.256| acc 0.916
Epoch  17 | train 0.213 | val 0.265| acc 0.916
Epoch  18 | train 0.199 | val 0.272| acc 0.916
Epoch  19 | train 0.194 | val 0.264| acc 0.915
Epoch  20 | train 0.193 | val 0.252| acc 0.922


In [12]:
# Save the Model
checkpoint = {
    'model_state': model.state_dict(),
    'mean' : torch.tensor(train_mean, dtype=torch.float32),
    'std' : torch.tensor(train_std, dtype=torch.float32)
}

torch.save(checkpoint, 'models/linear_nsynth.pth')

**Test Validation**

Load Model

In [13]:
VALIDATION_DATA_PATH = "nsynth-test"
VALIDATION_PROCESSED_DATA = "validation_data"


In [ ]:
#Generate Validate dataset
from load_validation import load_data

A,B = load_data()

np.savez(f"{VALIDATION_PROCESSED_DATA}/{VALIDATION_DATA_PATH}", X=A, Y=B)


In [ ]:
import numpy as np

# Validation Data on Linear Classification Trained on NSynth Data

checkpoint = torch.load('models/linear_nsynth.pth')

# Define model Architecture - hidden size must match the trained checkpoint
hidden = checkpoint['model_state']['0.weight'].shape[0]
model = nn.Sequential(
    nn.Linear(84, hidden), 
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(hidden, 88) # 88 = num of valid midi_notes
)

#d =  np.load(f"{VALIDATION_PROCESSED_DATA}/{VALIDATION_DATA_PATH}.npz")
d =  np.load(f"training_data/synth_train_data.npz")
X = torch.tensor(d['X'], dtype  = torch.float32)
Y = torch.tensor(d['Y'], dtype = torch.long)

#norm
X = (X - checkpoint['mean']) / checkpoint['std']

model.load_state_dict(checkpoint['model_state'])
model.eval()

with torch.no_grad():
    logits = model(X)

acc = (logits.argmax(1) == Y).float().mean().item()

print(f"val acc {acc:.3f}")

val acc 0.965
